#  Cloud nowcasting over Ireland notebook

**Goal:** look at the last 30 minutes of satellite images and predict **where the clouds will be** 15, 30 and 60 minutes from now.

We'll go through the whole pipeline, one step at a time:

1. **Collect data:** download satellite images (free, no login needed)
2. **Look at the data:** plot a satellite image
3. **Make labels:** turn temperatures into a "cloud / no cloud" map
4. **Build training examples:** pair *past images* with *future cloud maps*
5. **Build a neural network:** a small U-Net
6. **Train it**
7. **Evaluate it:** is it better than just guessing "nothing changes"?

**The data:** the European weather satellite **Meteosat (SEVIRI instrument)** takes a picture of Europe every 5 minutes.
It measures infrared "brightness temperature" in several channels. Cold = high cloud tops, warm = ground or sea.
Open Climate Fix keeps a free copy of years of these images on Google Cloud, so we'll use that.

**Packages you need:** `pip install numpy xarray zarr gcsfs ocf_blosc2 pyresample matplotlib torch`

In [1]:
!pip install numpy xarray zarr gcsfs ocf_blosc2 pyresample matplotlib torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 8.9 MB/s  0:00:009.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 14.5 MB/s  0:00:003.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 19.5 MB/s  0:00:008.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 19.9 MB/s  0:00:000.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44/44 [gcsfs]/44 [gcsfs]/44 [google-cloud-storage-control]


## 0. Imports and settings

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import ocf_blosc2 
from pyresample import create_area_def
from pyresample.area_config import load_area_from_string

torch.manual_seed(101) # For reproductibilty!!!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# Settings you can play with 

# Which days to download (one archive file per year, 2008-2022). Spreading days over the
# year gives the model both summer and winter weather. Each day takes ~20 s to download.
DAYS = pd.date_range("2021-01-05", "2021-12-31", freq="30D")   # 12 days
TEST_DAYS = 3            # the last 3 days are kept aside to test the model

# Satellite channels to use (all infrared / water vapour, so they work day and night)
CHANNELS = ["IR_039", "WV_062", "WV_073", "IR_087", "IR_108", "IR_120", "IR_134"]

HISTORY = 6              # model sees 6 frames = the last 30 minutes (one frame every 5 min)
LEADS = [3, 6, 12]       # predict 3, 6 and 12 frames ahead = +15, +30, +60 minutes
CLOUD_THRESHOLD_K = 270  # a pixel colder than 270 K (-3 °C) in channel IR_108 counts as cloud

DATA_FILE = Path("data/notebook_data.npz")   # downloaded data is saved here, so we only download once

## 1. Collect data

The archive is one huge [zarr](https://zarr.dev) dataset per year covering all of Europe.
`xarray` opens it *lazily*: nothing is downloaded until we ask for actual numbers.
So we first cut out a box around Ireland, then download only that part.

In [ ]:
ARCHIVE_URL = "gs://public-datasets-eumetsat-solar-forecasting/satellite/EUMETSAT/SEVIRI_RSS/v4/{year}_nonhrv.zarr"

archive = xr.open_zarr(ARCHIVE_URL.format(year=2021), storage_options={"token": "anon"})
archive

In [ ]:
# Find the rows/columns of the satellite grid that cover a lat/lon box around Ireland
# (lon -18..-2, lat 48..58: extra space to the west so we can see Atlantic weather coming in).
satellite_grid = load_area_from_string(archive.data.attrs["area"])
ireland_box = create_area_def("ireland", "EPSG:4326", area_extent=(-18, 48, -2, 58), resolution=0.05)
cols, rows = satellite_grid.get_area_slices(ireland_box)
print("rows:", rows, " columns:", cols)

In [ ]:
def download_day(day):
    # Download one day (288 images) of our channels over Ireland.
    ds = xr.open_zarr(ARCHIVE_URL.format(year=day.year), storage_options={"token": "anon"})
    part = ds.data.isel(y_geostationary=rows, x_geostationary=cols).sel(variable=CHANNELS)
    part = part.sel(time=slice(day, day + pd.Timedelta(hours=23, minutes=59)))
    images = part.values                             # shape (time, y, x, channel), values scaled to 0..1
    images = images.transpose(0, 3, 1, 2)            # -> (time, channel, y, x), the order PyTorch likes
    images = images[:, :, ::-1, ::-1]                # flip so north is up and west is left
    return pd.DatetimeIndex(part.time.values), images


if DATA_FILE.exists():
    saved = np.load(DATA_FILE)
    times, images = pd.DatetimeIndex(saved["times"]), saved["images"]
    print("Loaded saved data from", DATA_FILE)
else:
    all_times, all_images = [], []
    for day in DAYS:
        t, img = download_day(day)
        print(f"{day:%Y-%m-%d}: {len(t)} images")
        all_times.append(t)
        all_images.append(img.astype(np.float16))    # float16 = half the memory of float32
    times, images = all_times[0].append(all_times[1:]), np.concatenate(all_images)
    DATA_FILE.parent.mkdir(exist_ok=True)
    np.savez(DATA_FILE, times=times.values, images=images)

print("images shape (time, channel, y, x):", images.shape)

## 2. Look at the data

The archive stores every channel squeezed into the range 0..1. For `IR_108` we can convert back to
temperature in kelvin with a simple formula (the numbers come from the archive's documentation).

In [ ]:
IR108_MIN, IR108_MAX = 199.10002, 313.2767   # kelvin that 0 and 1 correspond to
ir108 = CHANNELS.index("IR_108")


def to_kelvin(scaled):
    return IR108_MIN + scaled.astype(np.float32) * (IR108_MAX - IR108_MIN)


i = 150  # pick any image
plt.figure(figsize=(10, 5))
plt.imshow(to_kelvin(images[i, ir108]), cmap="gray_r")   # reversed gray: cold clouds look white
plt.colorbar(label="IR_108 brightness temperature (K)")
plt.title(f"Satellite image at {times[i]:%Y-%m-%d %H:%M} UTC  (Ireland is in the right half)")
plt.show()

## 3. Make labels: cloud or no cloud?

First we shrink the images to half size (averaging 2×2 pixel blocks) so training is fast, even on a CPU.
Then a pixel is **cloud (1)** if IR_108 is colder than 270 K, else **clear (0)**.
This simple rule is our "truth" — the model's job is to predict what this map will look like in the future.

In [ ]:
def shrink(x):
    # Average 2x2 blocks. Also trims the size to a multiple of 4, which the U-Net needs later.
    h, w = x.shape[-2] // 8 * 4, x.shape[-1] // 8 * 4
    x = x[..., : h * 2, : w * 2].astype(np.float32)
    return x.reshape(*x.shape[:-2], h, 2, w, 2).mean(axis=(-3, -1))


X = np.stack([shrink(img) for img in images]).astype(np.float16)   # inputs  (time, channel, y, x)
clouds = (to_kelvin(X[:, ir108]) < CLOUD_THRESHOLD_K).astype(np.float16)   # labels (time, y, x)
X = np.nan_to_num(X)   # a few missing pixels -> 0

print("inputs:", X.shape, " cloud maps:", clouds.shape)
print(f"{clouds.mean():.0%} of all pixels are cloudy")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].imshow(X[i, ir108], cmap="gray_r"); ax[0].set_title("IR_108 (shrunk)")
ax[1].imshow(clouds[i], cmap="Blues_r"); ax[1].set_title("Cloud mask (white = cloud)")
plt.show()

## 4. Build training examples

One training example = **input:** 6 consecutive frames (30 min) × 7 channels = 42 image layers,
**target:** the cloud maps 15, 30 and 60 minutes after the last input frame.

We only use moments where all the needed frames exist and are exactly 5 minutes apart
(there are gaps at the edges of each day and when the satellite had problems).

In [ ]:
five_min = pd.Timedelta(minutes=5)
position = {t: k for k, t in enumerate(times)}   # time -> index in our arrays

good = []   # indices of "now" frames that have the full history and all future targets
for k, t in enumerate(times):
    needed = [t - n * five_min for n in range(HISTORY)] + [t + n * five_min for n in LEADS]
    if all(n in position for n in needed):
        good.append(k)
good = np.array(good)

# Split by DAY, not randomly: frames 5 minutes apart look almost identical, so a random split
# would let the model "cheat" by seeing near-copies of the test images during training.
test_start = DAYS[-TEST_DAYS]
train_idx = good[times[good] < test_start]
test_idx = good[times[good] >= test_start]
print(f"{len(train_idx)} training examples, {len(test_idx)} test examples")

In [ ]:
class CloudDataset(torch.utils.data.Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, n):
        k = self.indices[n]
        past = X[k - HISTORY + 1 : k + 1]                       # (6, 7, H, W)
        past = past.reshape(-1, *past.shape[-2:])               # stack into (42, H, W)
        future = np.stack([clouds[k + lead] for lead in LEADS]) # (3, H, W)
        now = clouds[k]                                         # current cloud map, used for the baseline
        return (torch.tensor(past, dtype=torch.float32),
                torch.tensor(future, dtype=torch.float32),
                torch.tensor(now, dtype=torch.float32))


# Neighbouring examples are almost the same, so using every 2nd one for training loses little and is 2x faster
train_loader = torch.utils.data.DataLoader(CloudDataset(train_idx[::2]), batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(CloudDataset(test_idx), batch_size=16)

past, future, now = next(iter(train_loader))
print("one batch  -> input:", tuple(past.shape), " target:", tuple(future.shape))

## 5. Build the neural network: a small U-Net

A **U-Net** is the classic network for "image in → image out" problems.

* The **encoder** (left side of the "U") shrinks the image step by step, so the network can see large areas
  and learn *how clouds move*.
* The **decoder** (right side) grows it back to full size to draw the prediction.
* **Skip connections** copy fine details from the encoder straight to the decoder, so the output stays sharp.

The output has 3 layers, one per lead time. Each pixel is a score; `sigmoid(score)` turns it into a
cloud probability between 0 and 1.

In [ ]:
def conv_block(c_in, c_out):
    # Two 3x3 convolutions, each followed by batch-norm and ReLU
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
    )


class SmallUNet(nn.Module):
    def __init__(self, in_channels, out_channels, width=32):
        super().__init__()
        self.down1 = conv_block(in_channels, width)          # full size
        self.down2 = conv_block(width, width * 2)            # 1/2 size
        self.bottom = conv_block(width * 2, width * 4)       # 1/4 size
        self.up2 = nn.ConvTranspose2d(width * 4, width * 2, 2, stride=2)
        self.dec2 = conv_block(width * 4, width * 2)
        self.up1 = nn.ConvTranspose2d(width * 2, width, 2, stride=2)
        self.dec1 = conv_block(width * 2, width)
        self.out = nn.Conv2d(width, out_channels, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(F.max_pool2d(d1, 2))
        b = self.bottom(F.max_pool2d(d2, 2))
        u2 = self.dec2(torch.cat([self.up2(b), d2], dim=1))   # skip connection from d2
        u1 = self.dec1(torch.cat([self.up1(u2), d1], dim=1))  # skip connection from d1
        return self.out(u1)


model = SmallUNet(HISTORY * len(CHANNELS), len(LEADS)).to(device)
print(f"The model has {sum(p.numel() for p in model.parameters()):,} trainable numbers (parameters)")

## 6. Train

Each training step:
1. the model makes a prediction for a batch of examples,
2. the **loss** (binary cross-entropy) measures how wrong it is,
3. `loss.backward()` works out how to change each parameter to reduce the loss,
4. the **optimizer** (Adam) makes that change.

One **epoch** = one pass over all training examples. The loss should go down.

In [ ]:
EPOCHS = 10
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0
    for past, future, _ in train_loader:
        past, future = past.to(device), future.to(device)
        prediction = model(past)
        loss = F.binary_cross_entropy_with_logits(prediction, future)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    losses.append(total / len(train_loader))
    print(f"epoch {epoch:2d}: loss {losses[-1]:.4f}")

plt.plot(range(1, EPOCHS + 1), losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("Training loss")
plt.show()

## 7. Evaluate on the test days

We need something to compare against. The simplest forecast is **persistence**: "the clouds will stay exactly
where they are now". It's surprisingly hard to beat for short lead times! Our model is only useful if it does better.

Two scores:
* **Accuracy:** fraction of pixels predicted correctly (cloud or clear).
* **IoU (intersection over union):** overlap between predicted cloud and real cloud, divided by their combined area.
  1.0 = perfect. It ignores the easy clear-sky pixels, so it's stricter.

In [ ]:
def scores(predicted, actual):
    # predicted, actual: boolean tensors, True = cloud
    accuracy = (predicted == actual).float().mean().item()
    iou = (predicted & actual).sum().item() / max((predicted | actual).sum().item(), 1)
    return accuracy, iou


model.eval()
model_preds, persistence_preds, actuals = [], [], []
with torch.no_grad():
    for past, future, now in test_loader:
        model_preds.append(torch.sigmoid(model(past.to(device))).cpu() > 0.5)
        persistence_preds.append((now > 0.5).unsqueeze(1).expand_as(future))  # same "now" map for every lead
        actuals.append(future > 0.5)
model_preds, persistence_preds, actuals = torch.cat(model_preds), torch.cat(persistence_preds), torch.cat(actuals)

print(f"{'lead':>8} | {'model acc':>9} {'persist acc':>11} | {'model IoU':>9} {'persist IoU':>11}")
for n, lead in enumerate(LEADS):
    m_acc, m_iou = scores(model_preds[:, n], actuals[:, n])
    p_acc, p_iou = scores(persistence_preds[:, n], actuals[:, n])
    print(f"{f'+{lead * 5} min':>8} | {m_acc:9.1%} {p_acc:11.1%} | {m_iou:9.3f} {p_iou:11.3f}")

Let's look at one forecast with our own eyes:

In [ ]:
n = len(test_idx) // 2           # pick a test example
past, future, now = test_loader.dataset[n]
with torch.no_grad():
    probability = torch.sigmoid(model(past[None].to(device)))[0].cpu()

lead = 2                         # 0 = +15 min, 1 = +30 min, 2 = +60 min
panels = [(now, "Now"), (future[lead], f"Actual +{LEADS[lead] * 5} min"),
          (now, f"Persistence +{LEADS[lead] * 5} min"), (probability[lead], f"U-Net +{LEADS[lead] * 5} min")]
fig, ax = plt.subplots(1, 4, figsize=(20, 3.5))
for a, (img, title) in zip(ax, panels):
    a.imshow(img, cmap="Blues_r", vmin=0, vmax=1)
    a.set_title(title); a.axis("off")
fig.suptitle(f"Forecast made at {times[test_idx[n]]:%Y-%m-%d %H:%M} UTC  (white = cloud)")
plt.show()

## 8. Save the trained model

In [ ]:
Path("models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "models/notebook_unet.pt")
print("Saved to models/notebook_unet.pt")

# To load it again later:
# model = SmallUNet(HISTORY * len(CHANNELS), len(LEADS))
# model.load_state_dict(torch.load("models/notebook_unet.pt"))

## Next steps / things to try

* **More data:** add more days to `DAYS` (e.g. `freq="10D"`, or several years). More data usually helps most.
* **Train longer:** increase `EPOCHS`.
* **Fewer channels:** try `CHANNELS = ["IR_108"]` — how much do the other channels help?
* **Bigger model:** `SmallUNet(..., width=64)`.
* **Live data:** the scripts in this folder (`download.py`, `preprocess.py`, `collect.py`) fetch today's images
  from EUMETSAT (needs a free account) so you can run the model on real-time weather.